# 09 · XGBoost "con noción de distancia"

Si la SNGP no le gana a XGBoost, una pregunta natural es: ¿y si le agregamos la distancia a XGBoost, que es el que mejor ordena? Tomamos su score y lo corregimos según qué tan lejos está cada solicitante de los datos de entrenamiento: cuanto más lejos, más se achica su logit hacia 0, o sea hacia "50% de mora" (no sé, no me confío). La fórmula es z = logit_XGB / raíz(1 + λ · distancia²), la misma idea del ajuste por varianza de la SNGP pero montada sobre el mejor orden.

Con λ = 0 es exactamente XGBoost, así que este modelo "contiene" a XGBoost: si la distancia no aporta, la validación elige λ = 0 y empata; si aporta, mejora.

- **Lee:** `outputs/datos_procesados.npz`, `outputs/ids_*.csv`, `outputs/preds/xgboost_*.csv` y `outputs/preds/sngp_vars_*.csv`.
- **Guarda:** `outputs/xgb_distancia_resultados.csv`.
- **En el informe:** el cociente 0,998 de la distancia kNN y la frase "con las dos medidas que probamos, el parámetro elegido en validación resulta ser 0" de la sección 4.1.

In [1]:
# Bloque 1. Cargamos todo y calculamos dos formas de medir la distancia:
#   (a) kNN: la distancia promedio a los 10 vecinos más cercanos de train, en el espacio de las 200 variables. No usa
#       ninguna red, es la medida más "cruda";
#   (b) la varianza de la SNGP, que es la distancia que mide la capa GP sobre la representación aprendida.
# Tarda un minuto por el cálculo de vecinos (usamos 50 mil filas de train como referencia).
import os
import time
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from conformal import conformal_select

SEED = 42
N_REF = 50_000            # filas de train usadas como "referencia" para los vecinos (con las 184 mil tarda demasiado)

datos = np.load("outputs/datos_procesados.npz", allow_pickle=True)   # las matrices del 01
X_tr = datos["X_train"].astype(np.float32)
X_va, y_va = datos["X_calib"].astype(np.float32), datos["y_calib"].astype(int)    # validación
X_po, y_po = datos["X_test"].astype(np.float32), datos["y_test"].astype(int)      # pool
ids_va = pd.read_csv("outputs/ids_calib.csv")["SK_ID_CURR"].to_numpy()
ids_po = pd.read_csv("outputs/ids_test.csv")["SK_ID_CURR"].to_numpy()

# Predicciones de XGBoost y de la SNGP, reordenadas para que la fila i sea el mismo solicitante que la fila i de X
xgb_va = pd.read_csv("outputs/preds/xgboost_calib.csv").set_index("SK_ID_CURR").loc[ids_va]
xgb_po = pd.read_csv("outputs/preds/xgboost_test.csv").set_index("SK_ID_CURR").loc[ids_po]
sn_va = pd.read_csv("outputs/preds/sngp_vars_calib.csv").set_index("SK_ID_CURR").loc[ids_va]
sn_po = pd.read_csv("outputs/preds/sngp_vars_test.csv").set_index("SK_ID_CURR").loc[ids_po]
assert (xgb_va["TARGET"].to_numpy() == y_va).all() and (xgb_po["TARGET"].to_numpy() == y_po).all(), "desalineado"
thin_va, thin_po = xgb_va["thin_file"].to_numpy().astype(bool), xgb_po["thin_file"].to_numpy().astype(bool)
lx_va, lx_po = xgb_va["logit"].to_numpy(), xgb_po["logit"].to_numpy()          # logit de default de XGBoost
var_va, var_po = sn_va["var"].to_numpy(), sn_po["var"].to_numpy()               # varianza SNGP

# Distancia kNN: promedio de la distancia a los 10 vecinos más cercanos de train
t0 = time.time()
# ref: 50 mil posiciones de train sorteadas sin repetir (replace=False); son los "vecinos posibles"
ref = np.random.default_rng(SEED).choice(len(X_tr), size=min(N_REF, len(X_tr)), replace=False)
# n_neighbors=10: cuántos vecinos mirar; algorithm="brute": calcula todas las distancias (en 200 dimensiones los
# atajos de árbol no ayudan); n_jobs=-1: usa todos los núcleos
nn = NearestNeighbors(n_neighbors=10, algorithm="brute", n_jobs=-1).fit(X_tr[ref])
# kneighbors devuelve (distancias, posiciones); [0] = las distancias, .mean(1) = promedio de los 10 por solicitante
d_va = nn.kneighbors(X_va)[0].mean(1)
d_po = nn.kneighbors(X_po)[0].mean(1)
d0 = np.median(d_va[~thin_va])                     # escala: distancia típica del "resto" en validación
d_va, d_po = d_va / d0, d_po / d0                  # distancia relativa (1 = típica)
print(f"vecinos calculados en {(time.time() - t0) / 60:.1f} min")
print("distancia kNN mediana thin / resto (pool):", round(float(np.median(d_po[thin_po]) / np.median(d_po[~thin_po])), 3))
print("varianza SNGP mediana thin / resto (pool):", round(float(np.median(var_po[thin_po]) / np.median(var_po[~thin_po])), 3))


def score_dist(logit, dist2, lam):
    """Prob. de buen pago con el logit encogido según la distancia (lam = 0: XGBoost puro).
    logit: logit de default de XGBoost. dist2: distancia al cuadrado de cada solicitante.
    lam: cuánto pesa la distancia (más grande = encoge más a los lejanos)."""
    z = logit / np.sqrt(1 + lam * dist2)           # lejos -> divisor grande -> z cerca de 0 (50% de mora)
    return 1 - 1 / (1 + np.exp(-z))                # 1 - sigmoide = prob. de BUEN pago (lo que usa el conformal)

# Distancia kNN mediana thin / resto en el pool: 0.998, y varianza SNGP mediana thin / resto: 1.124. Por ninguna de las
# dos medidas los thin-file están más lejos de los datos de entrenamiento que el resto (el 0,998 es uno de los cinco
# cocientes de la sección 4.1).
# Ojo: en 200 dimensiones las distancias kNN tienden a parecerse entre todos los puntos (la misma "maldición de la
# dimensionalidad" del 03), así que la varianza de la SNGP, que trabaja sobre una representación aprendida, es la
# medida más informativa de las dos.

vecinos calculados en 0.8 min
distancia kNN mediana thin / resto (pool): 0.998
varianza SNGP mediana thin / resto (pool): 1.124


In [2]:
# Bloque 2. Elegimos lambda (lam) EN VALIDACIÓN, sin mirar el pool, igual que en el 05: partimos validación 20 veces en
# calibración y prueba, corremos el conformal sobre thin-file y nos quedamos con el lam de mayor aprobación thin-file
# promedio entre alfa 3% y 9%. Una tabla para cada forma de medir la distancia; la fila lam = 0 es XGBoost puro.
ALFAS_VAL = [0.03, 0.05, 0.07, 0.09]
# Para cada medida: (distancia al cuadrado, valores de lam a probar). La varianza ya es una "distancia al
# cuadrado"; la multiplicamos por pi/8 para que sea la misma fórmula del ajuste mean-field del 05
GRILLAS = {"kNN": (d_va ** 2, [0, 0.1, 0.3, 1, 3]),
           "var SNGP": (np.pi / 8 * var_va, [0, 10, 30, 100, 300, 1000])}
estrato_v = 2 * thin_va.astype(int) + y_va         # 4 grupos (segmento x TARGET) para estratificar
elegido = {}                                       # medida -> lam elegido
for nombre, (dist2, lams) in GRILLAS.items():
    filas = []
    for r in range(20):                            # 20 particiones de validación en mitades
        i_cal, i_test = train_test_split(np.arange(len(y_va)), test_size=0.5, stratify=estrato_v, random_state=r)
        rng = np.random.default_rng(r)
        c, t = i_cal[thin_va[i_cal]], i_test[thin_va[i_test]]   # solo thin-file (por segmento)
        for lam in lams:
            mu = score_dist(lx_va, dist2, lam)
            for alfa in ALFAS_VAL:
                # .mean() de True/False = tasa de aprobación
                filas.append({"lam": lam, "alfa": alfa, "tasa": conformal_select(mu[c], y_va[c], mu[t], alfa, rng).mean()})
    tabla = pd.DataFrame(filas).pivot_table(index="lam", columns="alfa", values="tasa").round(3)
    tabla["promedio"] = tabla.mean(axis=1).round(3)  # criterio de elección fijado antes: promedio entre alfas
    elegido[nombre] = tabla["promedio"].idxmax()     # el lam de la fila con mayor promedio
    print(f"\n{nombre}:\n{tabla.to_string()}\nlam elegido: {elegido[nombre]}")

# Aprobación thin-file promedio en validación (alfa 3-9%):
#   kNN: lam 0 -> 0.640, 0.1 -> 0.635, 0.3 -> 0.626, 1 -> 0.596, 3 -> 0.565.
#   var SNGP: lam 0 -> 0.640, 10 -> 0.638, 30 -> 0.639, 100 -> 0.633, 300 -> 0.619, 1000 -> 0.586.
# La aprobación baja a medida que la distancia pesa más: penalizar por distancia castiga a solicitantes que no tenían
# por qué ser castigados. Se elige lam = 0 en los dos casos, o sea XGBoost + distancia = XGBoost.


kNN:
alfa   0.03   0.05   0.07   0.09  promedio
lam                                       
0.0   0.226  0.557  0.818  0.958     0.640
0.1   0.220  0.545  0.816  0.958     0.635
0.3   0.208  0.520  0.818  0.959     0.626
1.0   0.136  0.483  0.810  0.957     0.596
3.0   0.061  0.450  0.796  0.954     0.565
lam elegido: 0.0

var SNGP:
alfa   0.03   0.05   0.07   0.09  promedio
lam                                       
0     0.226  0.557  0.818  0.958     0.640
10    0.220  0.556  0.818  0.958     0.638
30    0.221  0.557  0.818  0.958     0.639
100   0.210  0.547  0.815  0.958     0.633
300   0.170  0.531  0.818  0.958     0.619
1000  0.131  0.446  0.811  0.957     0.586
lam elegido: 0


In [3]:
# Bloque 3. Con los lam ya elegidos, una sola evaluación en el pool: el experimento conformal thin-file de siempre
# (20 particiones), comparando XGBoost, XGBoost + distancia kNN, XGBoost + distancia var SNGP y la SNGP. Si los lam
# elegidos son 0, las filas de XGBoost + distancia tienen que dar igual que XGBoost.
lx_sn_po = sn_po["logit"].to_numpy()               # logit de la SNGP en el pool
# Nombre -> prob. de buen pago de cada modelo (el conformal usa solo el orden de estos números)
modelos = {"XGBoost": 1 - 1 / (1 + np.exp(-lx_po)),
           f"XGB + dist kNN (lam={elegido['kNN']})": score_dist(lx_po, d_po ** 2, elegido["kNN"]),
           f"XGB + dist var SNGP (lam={elegido['var SNGP']})": score_dist(lx_po, np.pi / 8 * var_po, elegido["var SNGP"]),
           "SNGP": 1 - 1 / (1 + np.exp(-lx_sn_po))}
ALFAS = [0.03, 0.05, 0.07, 0.09]
estrato = 2 * thin_po.astype(int) + y_po
filas = []
for r in range(20):
    i_cal, i_test = train_test_split(np.arange(len(y_po)), test_size=0.5, stratify=estrato, random_state=r)
    rng = np.random.default_rng(r)
    c, t = i_cal[thin_po[i_cal]], i_test[thin_po[i_test]]
    for nombre, mu in modelos.items():
        for alfa in ALFAS:
            a = conformal_select(mu[c], y_po[c], mu[t], alfa, rng)
            filas.append({"r": r, "Modelo": nombre, "alfa": alfa, "tasa": a.mean(),
                          "fdr": (a & (y_po[t] == 1)).sum() / max(a.sum(), 1)})   # morosos aprobados / aprobados
res = pd.DataFrame(filas)
tab = res.pivot_table(index="Modelo", columns="alfa", values="tasa").reindex(list(modelos)).round(3)
print("Tasa de aprobación thin-file (pool):\n", tab.to_string())
print("\nDiferencia contra XGBoost:\n", (tab - tab.loc["XGBoost"]).round(3).to_string())
print("\nFDR media (debería quedar <= alfa):\n",
      res.pivot_table(index="Modelo", columns="alfa", values="fdr").reindex(list(modelos)).round(3).to_string())
res.to_csv("outputs/xgb_distancia_resultados.csv", index=False)

# Las dos versiones de XGBoost + distancia dan igual que XGBoost (0.289 / 0.544 / 0.804 / 0.956 a alfa 3 / 5 / 7 / 9%),
# porque lam = 0 (el -0.001 de la versión kNN es solo un desempate distinto). La SNGP queda abajo (-6.3, -2.3, -2.1 y
# -0.2 puntos). La FDR se respeta en todos (queda <= alfa).
# Conclusión: ni siquiera agregándole la distancia al mejor modelo se gana algo en Home Credit, y no porque la SNGP
# sea débil: es que los thin-file no están lejos y no hay distancia que aprovechar.

Tasa de aprobación thin-file (pool):
 alfa                          0.03   0.05   0.07   0.09
Modelo                                                 
XGBoost                      0.289  0.544  0.804  0.956
XGB + dist kNN (lam=0.0)     0.288  0.544  0.804  0.956
XGB + dist var SNGP (lam=0)  0.289  0.544  0.804  0.956
SNGP                         0.226  0.521  0.783  0.954

Diferencia contra XGBoost:
 alfa                          0.03   0.05   0.07   0.09
Modelo                                                 
XGBoost                      0.000  0.000  0.000  0.000
XGB + dist kNN (lam=0.0)    -0.001  0.000  0.000  0.000
XGB + dist var SNGP (lam=0)  0.000  0.000  0.000  0.000
SNGP                        -0.063 -0.023 -0.021 -0.002

FDR media (debería quedar <= alfa):
 alfa                          0.03   0.05  0.07   0.09
Modelo                                                
XGBoost                      0.031  0.049  0.07  0.090
XGB + dist kNN (lam=0.0)     0.031  0.049  0.07  0.090
XGB